<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Data to Decisions: GPU-Accelerated Decision Optimization</b></h1>
<h2><b>Exercise 2:</b> Accelerated MIP Solver with cuOpt Python API</h2>
<br>

In this exercise, you will learn how to model a mixed-integer programming problem in the <b>supply chain optimization</b> context, using cuOpt's algebraic modeling API.

> **Student challenge version:** Replace every `<<<TO DO>>>` marker with working Python code, then run the cells in order. The completed reference notebook with the same filename lives one folder up.

> Tip: if a cell raises a syntax error, look for the next `<<<TO DO>>>` marker in that cell.

<hr>

# **Mixed-Integer Linear Programming**

Mixed-Integer Linear Programming (MILP) models express a decision optimization problem in the form of a linear objective function of decision variables `x_1, x_2, ...` and constraints also in the form of linear expressions of the same variables with `<=`, `=` or `>=` a right-hand side value. What makes a MILP a "mixed" model is the existence of both continuous and discrete (integer) variables.

Consider the following simple example model where the objective is to <b>maximize the total profit</b> earned from producing two products, with respective <b>decision variables</b> `x_1` and `x_2` representing quantities to be produced, while satisfying two <b>resource availabilty constraints</b>. In this case, one of the variables (`x_1`) assumes only integer or discrete values while the other variable can be continuous or fractional, hence the term "mixed-integer".

Notice that all mathematical expressions (objective function and constraints) are in linear combinations of these two decision variables.

**Optimization For Objective:**
> $\Large\text{max } z = 4 x_1 + 5 x_2$

**Subject to Constraints:**

> $\Large 2 x_1 + x_2 \leq 8$
> 
> $\Large x_1 + 2 x_2 \leq 7$
> 
> $\Large x_1,\ x_2 \geq 0$
>
> $\Large x_1$ **integer**

MILP models can be used to describe most real-world decision optimization problems and can quickly get very large in the number of decisions variables (or "columns") and constraints (or "rows"). The terms "columns" and "rows" refer to the underlying coefficient matrix `A` of the optimization problem that help represent the overall constraint set in a single matrix multiplication form `A x = b` where `A` is the coefficient matrix of rows and columns, `x` is the decision variable vector and `b` is the right-hand-side values of the constraints (`8` and `7` in the above example).

While the size of the problem in rows and columns and the presence of integer decision variables are important factors contributing to problem complexity, another contributing factor is the <b>sparsity of the matrix</b> `A`. cuOpt MIP Solver takes advantage of various memory management and parallel computing techniques on the GPU to overcome complexities associated with the size and structure of MILP models. But before discussing those, let's first get our hands dirty in modeling and solving a MILP model using cuOpt.

For further information on mixed-integer linear programming models and solution techniques, feel free to visit [here](https://en.wikipedia.org/wiki/Integer_programming).

## Problem Definition


Let's start by modeling a small supply chain production system that sources and packages cheese for consumers as a MILP problem. This system consists of <b>6 vendors</b> and <b>2 plants</b> where vendors are suppliers of either <b>high-quality</b> cheese scraps or <b>low-quality</b> cheese scraps and the plants procure cheese scraps from various vendors to meet their end-of-month demand of packaged cheese.
This system is subject to multiple restrictions like:

- Each vendor can supply to at most one plant.
- Maximum % of low-quality cheese in the packaged product should be 20%
- Total weight of all cheese scraps supplied by vendors to plant P should satisfy the demand at plant P
- Total weight of all cheese scraps supplied by vendor V to the plants should be less than or equal to the supply capacity of the vendor

Also provided is the shipping cost per pound of cheese scraps between each vendor-plant pair.

The aim of the problem is to <b>find the quantity of cheese scraps to be supplied by each vendor to the plants such that all above constraints are met at the lowest possible total shipping cost</b>.

Let's start by defining the supply-chain network parameters.

In [ ]:
# The production system has 6 vendors
Vendors = ['VH1', 'VH2', 'VH3', 'VH4', 'VL1', 'VL2']
# Vendors supplying high-quality cheese
Vendors_H = ['VH1', 'VH2', 'VH3', 'VH4']
# Vendors supplying low-quality cheese
Vendors_L = ['VL1', 'VL2']
# Max Supply Capacity of Cheese at each vendor
Q = {'VH1': 60, 'VH2': 20, 'VH3': 30, 'VH4': 40, 'VL1': 40, 'VL2': 20}

# The production system has 2 plants
Plants = ['P1', 'P2']
# Demand at each Plant
D = {'P1': 100, 'P2': 60}
# Max percent low-quality cheese allowed in final product is 20%
u = 0.20

# Cost to ship one pound of cheese scraps from vendor 𝑣 to plant 𝑝
C = {'VH1': { 'P1': 5, 'P2': 4 },
     'VH2': { 'P1': 2, 'P2': 2 },
     'VH3': { 'P1': 2, 'P2': 4 },
     'VH4': { 'P1': 3, 'P2': 2 },
     'VL1': { 'P1': 1, 'P2': 1 },
     'VL2': { 'P1': 1, 'P2': 2 }
    }

You can imagine how quickly this production system can get really large if hundreds or thousands of vendors, tens or hundreds of plants, and hundreds of products are included and the optimization needs to be carried out over not one but multiple planning periods.

<b>Let's now try to visualize this system with networkX!</b>

You will first define a Python function to plot the supply chain network given its parameters, and then call this function to generate the actual visualization.

In [ ]:
# Function to plot the supply-chain network

import networkx as nx
import matplotlib.pyplot as plt

def draw_network(V, P, Q, D, C, X=None, Y=None):

    V = V[::-1]
    P = P[::-1]

    # Create bipartite graph
    B = nx.DiGraph()

    # Add nodes
    B.add_nodes_from(V, bipartite=0)
    B.add_nodes_from(P, bipartite=1)

    # Add edges from C
    for src, targets in C.items():
        for dst, weight in targets.items():
            if Y is None or (Y is not None and round(Y[src][dst].getValue(),2)!=0):
                B.add_edge(src, dst, weight=weight)

    # Layout for bipartite graph
    pos = nx.bipartite_layout(B, V)

    # Node labels
    labels_left = Q
    labels_right = D
    labels = {**labels_left, **labels_right}

    # Draw nodes and edges
    plt.figure(figsize=(7, 5))
    nx.draw(
        B, pos,
        with_labels=True,
        labels={node: f"{node}\n({labels[node]} lbs)" for node in B.nodes()},
        node_color=["skyblue" if n in V else "lightgreen" for n in B.nodes()],
        node_size=1500,
        font_size=8,
        edge_color="gray"
    )

    # Draw edge labels
    if X is None:
        edge_labels = {(u, v): f"${d['weight']}/lb" for u, v, d in B.edges(data=True)}
    else:
        edge_labels = {(u, v): f"{round(X[u][v].getValue(),2)} lbs\n${round(d['weight'] * X[u][v].getValue(), 2)}" for u, v, d in B.edges(data=True)}
    nx.draw_networkx_edge_labels(B, pos, edge_labels=edge_labels, font_size=8)

    plt.axis("off")
    plt.show()

draw_network(Vendors, Plants, Q, D, C)

Great! Now that we have the problem well-defined, we are ready to model and optimize our problem with cuOpt!

<hr>

## **Step 1:** Initialize the Optimization Problem

Let us name our model **"Cheese Manufacturing MILP"** and create a `Problem` object.

In [ ]:
import cuopt
from cuopt.linear_programming.problem import Problem, CONTINUOUS, INTEGER, MINIMIZE

# TODO 02.1: Initialize the MILP model.
problem = <<<TO DO>>>


<hr>

## **Step 2:** Add Variables

First we need to add the <b>decision variables</b>. Variables represent the unknowns in the problem that cuOpt seeks to determine via optimization so as to minimize the total shipping cost. In this MILP problem we have `CONTINUOUS` and `INTEGER` variables.

1) <b>$X_{v,p}$</b> - The non-negative amount (in pounds) of cheese scraps shipped from vendor `v` to plant `p`. These variables are of type `CONTINUOUS`.

> $X_{v,p} \geq 0$

2) <b>$Y_{v,p}$</b> - Binary variable denoting whether or not vendor `v` ships to plant `p`. The value of this variable can evaluate to either 0 or 1. These variables are of type `INTEGER` and to indicate that they are actually binary, we set lower and upper bounds on the variables as below:

> $0 \leq Y_{v,p} \leq 1$


In [ ]:
def variable_to_string(v):
    return (
        f"{v.getVariableName()} ({str(var.getVariableType())[6:].lower()}"
        f" variable in {[var.getLowerBound(), var.getUpperBound()]})"
    )
    
# Add Xv,p: continuous amount shipped from vendor V to plant P.
X = {}
for V in Vendors:
    X[V] = {}
    for P in Plants:
        varname = "x_"+ V + "_" + P
        X[V][P] = (problem.getVariable(varname) 
            or problem.addVariable(vtype=<<<TO DO>>>, name=varname))

# Add Yv,p: binary/integer decision for whether vendor V serves plant P.
Y = {}
for V in Vendors:
    Y[V] = {}
    for P in Plants:
        varname = "y_"+ V + "_" + P
        Y[V][P] = (problem.getVariable(varname) 
            or problem.addVariable(vtype=<<<TO DO>>>, lb=0, ub=1, name=varname))

for var in problem.getVariables():
    print(" - Added", variable_to_string(var))


Notice the ```vtype=CONTINUOUS``` and ```vtype=INTEGER``` designations for the two sets of variables.

All together, we have `6 x 2 = 12` continuous `X` and `6 x 2 = 12` binary `Y` variables. If you had 200 vendors and 10 plants and 3 types of products to be shipped, you would have had `200 x 10 x 3 = 6000` `X` and `200 x 10 x 3 = 6000` binary `Y` variables.

<hr>

## **Step 3:** Add Constraints

Next, we need to add the **supply-chain constraints** that impose limitations on these decision variables. Our production system has the following constraints that need to be modeled:

1) Each vendor should be assigned to at most one plant.

2) Sum of all cheese scrap amounts (in pounds) shipped from vendors to plant `p` should be equal to the demand (D) at plant `p`.

3) Amounts (in pounds) supplied by vendor `v` (`X_v,p`) cannot exceed the capacity (Q) at vendor `v`

4) Max percentage of low-quality cheese received at each plant should be 20%

#### **Understanding Component Arithmetic**

To create a constraint in `cuOpt`, you *could* construct them using the default constructors:

In [ ]:
# ?cuopt.linear_programming.problem.Variable
# ?cuopt.linear_programming.problem.LinearExpression 
# ?cuopt.linear_programming.problem.Constraint

# ?problem.addVariable
# ?problem.addConstraint

Alternatively, you can rely on implicit component arithmetic and coersions which can be used to enforce a more intuitive syntax:

In [ ]:
## Various operations are supported between Variables and right-hand-side arguments
## This includes basic arithmetic terms which help to build "Expressions"
# ??cuopt.linear_programming.problem.Variable.__add__
# ??cuopt.linear_programming.problem.Variable.__sub__

## These expressions can also be built onto in much the same way.
# ?cuopt.linear_programming.problem.LinearExpression
# ??cuopt.linear_programming.problem.LinearExpression.__add__

## There is also support for >=/<=/== operations, which create "Constraints"
# ?cuopt.linear_programming.problem.Constraint
# ??cuopt.linear_programming.problem.Variable.__eq__
# ??cuopt.linear_programming.problem.LinearExpression.__le__
# ??cuopt.linear_programming.problem.LinearExpression.__ge__
# ??cuopt.linear_programming.problem.LinearExpression.__eq__

## Note how gt/lt are not overridden. This means inclusive comparisons is required 
# ??cuopt.linear_programming.problem.LinearExpression.__gt__

Using this logic, we can create constraints in a much more intuitive manner:

In [ ]:
def constraint_to_string(constraint):
    # Format the left-hand side and right-hand side
    terms = [f"({coeff:g}*x{idx})" for idx, coeff in constraint.vindex_coeff_dict.items()]
    lhs = " + ".join(terms) if terms else "0"
    rhs = f"{constraint.getRHS():g}"
    
    # Translate sense enum to math symbol
    sense_map = {"L": "<=", "LE": "<=", "E": "==", "EQ": "==", "G": ">=", "GE": ">="}
    sense = sense_map.get(constraint.getSense())

    class_name = constraint.__class__.__name__
    const_name = constraint.getConstraintName() or "<unnamed>"
    return f"{class_name} {const_name} = [{lhs} {sense} {rhs}]"

sample_var = Y['VH1']['P1']
sample_linexp = Y['VH1']['P1'] + Y['VH1']['P2']
sample_constr = Y['VH1']['P1'] + Y['VH1']['P2'] <= 5

print(f"{sample_var = }")
print(f"{sample_linexp = }")
print(f"{sample_constr = }")
print("Created", constraint_to_string(sample_constr))

Now that we understand how we can create constraints, let's set them up for our problem.

#### **Adding Constraint Equation 1**

In [ ]:
c1_end = len(Vendors)
c1_consts = []
for i, V in enumerate(Vendors):
    const_name = f"C_{i}"
    # TODO 02.2: Add the vendor assignment limit.
    const_linexp = <<<TO DO>>>
    c1_consts += [problem.getConstraint(const_name) or
        problem.addConstraint(const_linexp, name=const_name)]

for const in c1_consts:
    print(" - Added", constraint_to_string(const))


This means the sum of binary "assignments" from each vendor `v` to all plants will be less than or equal to 1. Since `Y` variables are binary, this constraint also allows for the possibility that a vendor `v` may not supply any cheese scraps at all (if the sum turns out to be 0 in the optimal solution).

#### **Adding Constraint Equation 2**

In [ ]:
c2_end = c1_end + len(Plants)
c2_consts = []
for i, P in enumerate(Plants):
    const_name = f"C_{i + c1_end}"
    # TODO 02.3: Add the plant demand requirement.
    const_linexp = <<<TO DO>>>
    c2_consts += [problem.getConstraint(const_name) or
        problem.addConstraint(const_linexp, name=const_name)]

for const in c2_consts:
    print(" - Added", constraint_to_string(const))


That is, total cheese scraps (in pounds) supplied to each plant `p` must exactly match that plant's demand.

#### **Adding Constraint Equation 3**

In [ ]:
c3_end = c2_end + len(Plants) * len(Vendors)
c3_consts = []
idx = c2_end
for V in Vendors:
    for P in Plants:
        const_name = f"C_{idx}"
        # TODO 02.4: Add the shipment-assignment linking rule.
        const_lineq = <<<TO DO>>>
        c3_consts += [problem.getConstraint(const_name) or
            problem.addConstraint(const_lineq, name=const_name)]
        idx += 1

for const in c3_consts:
    print(" - Added", constraint_to_string(const))


This constraint says that the amount `X` shipped from vendor `v` to plant `p` cannot exceed supply capacity `Q` of vendor `v`. But notice that the right-hand side of the inequality constraint also includes `Y` binary variable as a factor. This means vendor `v` can ship to plant `p` within its capacity only if the optimization allows it to do so, as indicated when `Y_v,p = 1`. Otherwise, vendor `v` cannot ship to plant `p` even if it has capacity.

#### **Adding Constraint Equation 4**

In [ ]:
# L/(H+L) <= u i.e. (1-u)L <= uH
# where, u = 0.2
#        H = High-quality cheese scraps
#        L = Low-quality cheese scraps

u = 0.2

c4_end = c3_end + len(Plants)
c4_consts = []
for i, P in enumerate(Plants):
    const_name = f"C_{c3_end + i}"
    # TODO 02.5: Build the quality-mix constraint.
    rhs = <<<TO DO>>>
    lhs = <<<TO DO>>>
    const_lineq = (rhs <= lhs)
    c4_consts += [problem.getConstraint(const_name) or
        problem.addConstraint(const_lineq, name=const_name)]

for const in c4_consts:
    print(" - Added", constraint_to_string(const))


We want each plant `p` to have a ratio of total low quality cheese supplied from all vendors (`L`) to total cheese supplied (`H+L`) to the plant `<=` 0.2. If the expression `L / (H + L) <= u` where `u = 0.2` is "linearized", we have `(1 - u) L <= u H`. Also notice we have used the right vendor sets `Vendors_L` and `Vendors_H` to track low and high quality cheese scraps shipped.

<hr>

## **Step 4:** Set the Objective

The objective represents the goal of the optimization. The objective may be about minimizing cost, maximizing profit, minimizing resources (eg. vehicles, machines, personnel) employed, etc. depending on the context.

For this supply chain system, we want to minimize the total shipping cost from vendors to plants. Of course we would like to achieve this while simultaneously satisfying all the constraints of the problem.

In [ ]:
# TODO 02.6: Build the total transportation cost expression.
objective_exp = <<<TO DO>>>
# TODO 02.7: Set the optimization goal.
problem.<<<TO DO>>>


<hr>

## **Step 5**: Optimize

We have successfully modeled our problem in cuOpt. The final step is to solve it using cuOpt MIP solver!

In [ ]:
# TODO 02.8: Solve the MILP.
problem.<<<TO DO>>>


<hr><br>

## **Step 6**: Analyze The Results

Now that we have optimized the model such that the production system operates at minimal cost, let us take a look at the details of the solution.

In [ ]:
print("|==================================================================|")
print("|                        Solution Metadata                         |")
print("|==================================================================|")
solve_time = problem.SolveTime
primal_objective_value = problem.ObjValue
print(f" Optimal Solution found in {solve_time:.3f} seconds")
print(f" Minimized cost of operating the production system is {primal_objective_value:.2f}\n")

print("|==================================================================|")
print("|                    Optimized Variable Values                     |")
print("|==================================================================|")
for V in Vendors:
    for P in Plants:
        x = X[V][P].Value
        y = Y[V][P].Value
        if round(y, 2) != 0:
            print(f" Vendor {V} ships {round(x, 2)} lbs of cheese to Plant {P} ")

The output tells us that vendors 2, 3 and 4 collectively ship high quality cheese to plant 1 and vendor 1 only ships high quality cheese to plant 2, to satisfy total demand at these plants together with the low quality cheese shipped by vendors 1 and 2.

cuOpt solved this problem very fast (sub-second) since it's rather a toy problem. You will work with a larger problem in the next section and get a better sense of how long an optimization run can last when the problem is larger and potentially more complex.

Let's also visualize this solution with the same function we created above:


In [ ]:
draw_network(Vendors, Plants, Q, D, C, X, Y)

As a last step, we can save the model we created above so we can use it directly in the next exercise:

In [ ]:
# TODO 02.9: Save the small model for the LP notebook.
problem.<<<TO DO>>>


<hr>

# **Optimizing a Larger Problem**

Now that we are familiar with the modeling aspects of cuOpt, we can explore larger real-world end-to-end supply chain models that involve many more vendors, plants, products, timelines, inventory, etc. These models are typically stored and represented as an MPS (Mathematical Programming System) file.

The <b>[MPS format](https://en.wikipedia.org/wiki/MPS_(format))</b> is a widely used text-based file format for representing mathematical programming models, particularly linear programming (LP) and mixed-integer linear programming (MILP).

Let us try to solve a larger version of the problem we just solved with more variables and constraints to model additional concepts with this supply chain system. This version of the model indeed represents a real-world production system, with 448 variables and 396 constraints, imported from an MPS file - [**data/Cheese.mps**](./data/Cheese.mps) - courtesy of NVIDIA cuOpt partner [**SimpleRose, Inc.**](https://simplerose.com/).

**Disclaimer:** This problem introduces many different types of continuous and integer variables and many additional constraints to model different aspects of the supply chain system. We will not directly define this model using cuOpt like we did above, but will directly use an MPS file instead. If you are interested in reading more about the structure introduced by this MILP model, unhide the text below.

<details>
<summary><b>Click to read the details of the mathematical model</b></summary>

### Sets and indices

- 𝑉 Set of all vendors
- 𝑉𝑝 Set of vendors who can send cheese scraps to plant 𝑝
- 𝑉𝐻𝑝 Set of high-quality vendors who can send cheese scraps to plant 𝑝
- 𝑉𝐿𝑝 Set of low-quality vendors who can send cheese scraps to plant 𝑝
- 𝑃 Set of all plants
- 𝑃𝑉 Set of plants that can receive scraps from vendor 𝑣
- 𝑀 Set of months in the planning horizon (1 to 12)
- 𝑄 Set of scrap cheese qualities (H = high-quality, L = low-quality)

### Parameters

- D𝑝,𝑚 Demand at plant 𝑝 in month 𝑚 (dry pounds)
- S𝑝 Minimum end-of-month inventory level of high-quality cheese at plant 𝑝
- C𝑝 Pallet capacity at plant 𝑝
- 𝛿𝑝,𝑚 Penalty for unmet demand at plant 𝑝 in month 𝑚
- 𝜎𝑝,𝑚 Penalty for high-quality inventory shortfall at plant 𝑝 in month 𝑚
- 𝜏𝑝,𝑚 Penalty for exceeding pallet capacity at plant 𝑝 in month 𝑚
- 𝑐𝑣,𝑝 Cost to ship one pound of cheese scraps from vendor 𝑣 to plant 𝑝
- 𝜙𝑣,𝑚 Availability, in wet pounds, of cheese from vendor 𝑣 in month 𝑚
- 𝜓𝑣 Conversion factor to pallet positions per pound of cheese scraps from vendor 𝑣, includes stackability
of vendor’s pallet
- 𝜋 Yield factor representing the proportion of wet cheese scraps that remains after dehydration, used to
- convert input weight to dry finished product
- 𝜇 Max percent of low quality cheese allowed in final product at plant 𝑝

### Decision Variables

- 𝑥𝑣,𝑝,𝑚 Pounds of cheese from vendor 𝑣 used at plant 𝑝 in month 𝑚
- 𝑦𝑣,𝑝 Binary variable, 1 if vendor 𝑣 is assigned to plant 𝑝 for the year, 0 otherwise
- 𝑑𝑝,𝑚 Demand shortfall at plant 𝑝 in month 𝑚
- 𝑠𝑝,𝑚 Inventory shortfall of high-quality cheese at plant 𝑝 in month 𝑚
- 𝑡𝑝,𝑚 Pallet overage at plant 𝑝 in month 𝑚

### Objective Function

---

The objective of the strategic optimization model is to minimize the total cost of the system across the planning horizon.
The first component of the objective function represents the total freight cost, calculated based on which vendors are
assigned to which plants and the corresponding monthly cheese scrap availability. The second component penalizes
soft constraint violations, including unmet production demand, insufficient ending inventory of high-quality cheese
scraps, and excess inventory beyond plant pallet capacity. These penalty terms allow the model to remain feasible under
practical limitations while discouraging undesirable outcomes.

```
  min  =   ∑︁   ∑︁   ∑︁   𝑐𝑣,𝑝  𝜙𝑣,𝑚  𝑦𝑣,𝑝  +  ∑︁   ∑︁   𝛿𝑝,𝑚  𝑑𝑝,𝑚  +  𝜎𝑝,𝑚  𝑠𝑝,𝑚  +  𝜏𝑝,𝑚 𝑡𝑝,𝑚
          𝑝∈𝑃 𝑣∈𝑉𝑝 𝑚∈𝑀                    𝑝∈𝑃 𝑚∈𝑀
```

---

<h3>Constraint Equations</h3>

---

1. Equation 1 limits each vendor to being assigned to at most one plant.
    ```
      ∑︁  𝑦𝑣,𝑝  ≤1  ∀𝑣 ∈𝑉
     𝑝∈𝑃𝑣
    
    ```

    ---

2. Equation 2 ensures that forecasted production demand is satisfied at each plant in each month. The left-hand side
represents the total usable cheese scraps allocated to the plant from assigned vendors, and 𝑑𝑝,𝑚 captures any shortfall
between available input and required production. This variable appears in the objective function as a penalty term,
allowing for demand to be underfilled if necessary, but discouraging it through cost. The constraint is formulated as an
equality to explicitly account for any unmet demand.
    ```
      𝜋 ∑︁  𝑥𝑣,𝑝,𝑚  +  𝑑𝑝,𝑚  =  D𝑝,𝑚  ∀𝑝 ∈𝑃, 𝑚 ∈𝑀
        𝑣∈𝑉𝑃
    ```

    ---

3. Equation 3 ensures that a minimum level of high-quality inventory is maintained at each plant at the end of
every month. This constraint enforces a planning buffer to protect against production disruptions. The variable 𝑠𝑝,𝑚
captures any shortfall relative to the required inventory threshold 𝜎𝑝 and is penalized in the objective function. This
soft constraint allows the model to temporarily dip below the target inventory level if necessary, but discourages doing
so through cost.
    ```
              𝑛
        ∑︁     ∑︁  𝑦𝑣,𝑝  𝜙𝑣,𝑚  −  𝑥𝑣,𝑝,𝑚  − 𝑠𝑝,𝑚  ≥  S𝑝   ∀𝑝 ∈𝑃,𝑛 ∈𝑀
     𝑣∈𝑉𝐻𝑝 𝑚=1
    ```

    ---

4. Equation 4 ensures that total pallet usage at each plant does not exceed its available storage capacity. This includes
all incoming cheese scraps (converted to pallet equivalents) minus the amount consumed in production, summed over
time. The parameter 𝜓𝑣 converts wet pounds to pallet positions based on vendor-specific characteristics, and 𝜙𝑣,𝑚
represents the quantity of cheese available. If capacity must be exceeded in a given month, the variable 𝑡𝑝,𝑚 captures
the overage amount. This soft constraint allows for occasional overcapacity when necessary but penalizes it in the
objective function.
    ```
            𝑛
        ∑︁   ∑︁   𝜓𝑣  (𝑦𝑣,𝑝  𝜙𝑣,𝑚  − 𝑥𝑣,𝑝,𝑚) − 𝑡𝑝,𝑚  ≤  C𝑝   ∀𝑝 ∈𝑝,𝑛 ∈𝑀
     𝑣∈𝑉𝑝 𝑚=1
    ```

    ---

5. Equation 5 ensures that the cumulative amount of cheese scraps used from a vendor up to any month does not
exceed the cumulative amount available, and only if the vendor is assigned to that plant.
    ```
        𝑛             𝑛
        ∑︁  𝑥𝑣,𝑝,𝑚  ≤  ∑︁  𝜙𝑣,𝑚  𝑦𝑣,𝑝   ∀𝑣 ∈𝑉, 𝑝 ∈𝑃,𝑛 ∈𝑀
     𝑚=1          𝑚=1
    ```

    ---

6. Equation 6 enforces the blending constraint on low-quality cheese scraps by limiting their proportion in the final
product. To maintain product quality, the total amount of low-quality scraps used must not exceed a specified fraction
𝜇 of the total cheese scraps used. Although the original blending ratio constraint is non-linear, the formulation is
linearized as:
    ```
      𝐿 /(𝐻 + 𝐿)  ≤  𝜇   ===>   (1 − 𝜇𝑝)𝐿 ≤ 𝜇𝐻
    ```
    where 𝐿 is the amount of low-quality scraps used and 𝐻 is the amount of high-quality scraps used. The constraint
    ensures that the contribution of low-quality scraps to the production process at any plant remains within the allowed
    threshold.
    ```
                    𝑛                     𝑛
     (1 − 𝜇) ∑︁    ∑︁  𝑥𝑣,𝑝,𝑚  ≤  𝜇  ∑︁    ∑︁    𝑥𝑣,𝑝,𝑚   ∀𝑝 ∈𝑃,𝑚 ∈𝑀
             𝑣∈𝑉𝐿𝑝 𝑚=1             𝑣∈𝑉𝐸𝑝 𝑚=1
    ```

</details>

<hr>

## **Step 1:** Read the MPS file and initialize the problem

To get started, let's load in our model from the MPS file query it to get a sense of some its characteristics:

In [ ]:
# TODO 02.10: Load the larger production-planning model.
big_problem = <<<TO DO>>>


In [ ]:
print(f"""The supply-chain problem has:
 > {big_problem.NumVariables} variables
 > {big_problem.NumConstraints} constraints
 > {big_problem.NumNZs} non-zeros""")

<hr>

## **Step 2:** Optimize

We can now directly run cuOpt MIP solver to optimize this model.

In [ ]:
# TODO 02.11: Solve the larger MILP.
big_problem.<<<TO DO>>>


<hr>

**That was easy! Lets take a peek at the Solution:**

In [ ]:
print("|==================================================================|")
print("|                        Solution Metadata                         |")
print("|==================================================================|")
solve_time = big_problem.SolveTime
primal_objective_value = big_problem.ObjValue
print(f" Optimal Solution found in {solve_time:.3f} seconds")
print(f" Minimized cost of operating the production system is {primal_objective_value:.2f}\n")

print("|==================================================================|")
print("|                    Optimized Variable Values                     |")
print("|==================================================================|")
value_dict = dict()
for v in big_problem.getVariables():
    approx_value = round(v.Value, 7)  ## <- TODO: Try increasing rounding place
    value_dict[approx_value] = value_dict.get(approx_value, []) + [v.VariableName]

for value, keys in value_dict.items():
    print(f" * {len(keys):3.0f} variables have value close to {value}")
    if (preview := 0):                ## <- TODO: set preview := 10?
        print(" >  ", "|".join(keys[:preview]), "..." if len(keys)>preview else "", "\n")

MIP solver uses several techniques under the hood to find a solution to this (and other much larger-scale) problems:
- **MIP Heuristics algorithm:** a heuristic algorithm that generates pools of good feasible solutions and applies various search techniques to generate yet more alternative feasible solutions. This algorithm runs on the GPU.
- **Branch & Bound (B&B) algorithm:** cuOpt creates a B&B tree that branches off integer variables with fractional values, and explores this tree to solve various versions of the original problem with additional branching constraints. More to read on B&B [here](https://en.wikipedia.org/wiki/Branch_and_bound). This algorithm runs on the CPU.
- **PDLP solver:** The root node of the MIP B&B tree (the original problem) can be "relaxed" to allow integer variables to have fractional values. This not only makes the problem easier to solve but also provides a lower or upper bound for the original MIP problem objective and helps with pruning the branches of the B&B tree. We will talk more about this in the next exercise.

**Congratulations!** You finished this exercise by modeling a small MIP problem with cuOpt API and also loading a larger MIP problem in MPS format and solving it.

**In the next exercise, you will see how you can use cuOpt PDLP solver to solve LP version of the MIP problem you solved.**

<img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>